
## Folder 05 — File 02: Completeness and Uniqueness Checks:


**Purpose**: Measure how complete and unique the seven Silver datasets are.

This notebook focuses on two fundamental data-quality dimensions:
- **Completeness**: Are required fields populated?
- **Uniqueness**: Are business keys unique, without duplicate records?


It also calculates **null percentages, blank-value percentages, duplicate rows**, and **duplicate business-key groups**.

In [0]:
## 1. Import libraries

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType,
    TimestampType
)

from datetime import datetime

print("Libraries imported successfully.")


## 2. Project configuration

CATALOG = "workspace"
SILVER_SCHEMA = "ecommerce_silver"

REPORT_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}."
    "completeness_uniqueness_report"
)

DATASET_CONFIG = {

    "silver_customers": {
        "key_columns": ["customer_id"],
        "required_columns": [
            "customer_id",
            "first_name",
            "last_name",
            "email",
            "created_at"
        ]
    },

    "silver_products": {
        "key_columns": ["product_id"],
        "required_columns": [
            "product_id",
            "product_name",
            "category",
            "unit_price",
            "unit_cost",
            "created_at"
        ]
    },

    "silver_orders": {
        "key_columns": ["order_id"],
        "required_columns": [
            "order_id",
            "customer_id",
            "order_timestamp",
            "order_status",
            "order_total"
        ]
    },

    "silver_order_items": {
        "key_columns": ["order_item_id"],
        "required_columns": [
            "order_item_id",
            "order_id",
            "product_id",
            "quantity",
            "unit_price",
            "discount_rate",
            "line_total"
        ]
    },

    "silver_payments": {
        "key_columns": ["payment_id"],
        "required_columns": [
            "payment_id",
            "order_id",
            "payment_method",
            "payment_status",
            "amount",
            "payment_timestamp"
        ]
    },

    "silver_shipments": {
        "key_columns": ["shipment_id"],
        "required_columns": [
            "shipment_id",
            "order_id",
            "carrier",
            "shipment_status",
            "shipped_date"
        ]
    },

    "silver_inventory": {
        "key_columns": [
            "product_id",
            "warehouse_id",
            "snapshot_date"
        ],
        "required_columns": [
            "inventory_snapshot_id",
            "product_id",
            "warehouse_id",
            "quantity_on_hand",
            "snapshot_date"
        ]
    }
}


## 3. Create report schema

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS "
    f"`{CATALOG}`.`{SILVER_SCHEMA}`"
)

report_schema = StructType([

    StructField("dataset_name", StringType(), False),
    StructField("check_type", StringType(), False),
    StructField("column_name", StringType(), True),

    StructField("total_records", LongType(), False),
    StructField("failed_records", LongType(), False),

    StructField("failure_percentage", DoubleType(), False),

    StructField("status", StringType(), False),
    StructField("details", StringType(), True),

    StructField("checked_at", TimestampType(), False)
])


## 4. Reusable report function

def add_report(
    results,
    dataset_name,
    check_type,
    column_name,
    total_records,
    failed_records,
    details
):

    failure_percentage = (
        round(
            failed_records / total_records * 100,
            2
        )
        if total_records > 0
        else 0.0
    )

    if total_records == 0:
        status = "FAIL"
    elif failed_records == 0:
        status = "PASS"
    else:
        status = "FAIL"

    results.append({

        "dataset_name": dataset_name,

        "check_type": check_type,

        "column_name": column_name,

        "total_records": int(total_records),

        "failed_records": int(failed_records),

        "failure_percentage": float(failure_percentage),

        "status": status,

        "details": details,

        "checked_at": datetime.now()
    })


## 5. Define metadata columns to exclude

METADATA_COLUMNS = [
    "_source_file",
    "_source_file_modification_time",
    "_ingestion_timestamp",
    "_ingestion_date",
    "_rescued_data"
]


## 6. Run completeness and uniqueness checks

all_results = []

for dataset_name, config in DATASET_CONFIG.items():

    table_name = (
        f"{CATALOG}.{SILVER_SCHEMA}.{dataset_name}"
    )

    print("=" * 60)
    print(f"Checking dataset: {dataset_name}")
    print("=" * 60)

    if not spark.catalog.tableExists(table_name):
        raise Exception(
            f"Silver table not found: {table_name}"
        )

    df = spark.table(table_name)

    total_records = df.count()

    available_columns = set(df.columns)

    required_columns = config["required_columns"]

    key_columns = config["key_columns"]

    # --------------------------------------------------
    # Check 1: Table record count
    # --------------------------------------------------

    add_report(
        all_results,
        dataset_name,
        "total_record_count",
        None,
        total_records,
        0 if total_records > 0 else 1,
        "Table must contain at least one record."
    )

    # --------------------------------------------------
    # Check 2: Required columns exist
    # --------------------------------------------------

    missing_columns = [
        column
        for column in required_columns
        if column not in available_columns
    ]

    add_report(
        all_results,
        dataset_name,
        "required_columns",
        None,
        len(required_columns),
        len(missing_columns),
        f"Missing columns: {missing_columns}"
    )

    if missing_columns:
        print(f"Missing required columns: {missing_columns}")
        continue

    # --------------------------------------------------
    # Check 3: Null and blank values per column
    # --------------------------------------------------

    for column in required_columns:

        missing_condition = (
            F.col(column).isNull()
            |
            (
                F.trim(
                    F.col(column).cast("string")
                ) == ""
            )
        )

        missing_count = (
            df
            .filter(missing_condition)
            .count()
        )

        add_report(
            all_results,
            dataset_name,
            "null_and_blank_check",
            column,
            total_records,
            missing_count,
            f"Missing or blank values in {column}."
        )

    # --------------------------------------------------
    # Check 4: Business-key null values
    # --------------------------------------------------

    null_key_condition = None

    for key in key_columns:

        condition = F.col(key).isNull()

        null_key_condition = (
            condition
            if null_key_condition is None
            else null_key_condition | condition
        )

    null_key_records = (
        df
        .filter(null_key_condition)
        .count()
    )

    add_report(
        all_results,
        dataset_name,
        "null_business_key_check",
        ",".join(key_columns),
        total_records,
        null_key_records,
        "Records containing at least one null business-key value."
    )

    # --------------------------------------------------
    # Check 5: Duplicate business-key groups
    # --------------------------------------------------

    non_null_key_df = df.filter(
        ~null_key_condition
    )

    duplicate_key_groups_df = (
        non_null_key_df
        .groupBy(*key_columns)
        .count()
        .filter(F.col("count") > 1)
    )

    duplicate_key_groups = (
        duplicate_key_groups_df.count()
    )

    duplicate_extra_rows = (
        duplicate_key_groups_df
        .agg(
            F.sum(F.col("count") - 1).alias("extra_rows")
        )
        .first()["extra_rows"]
    )

    duplicate_extra_rows = (
        int(duplicate_extra_rows)
        if duplicate_extra_rows is not None
        else 0
    )

    add_report(
        all_results,
        dataset_name,
        "duplicate_business_key_groups",
        ",".join(key_columns),
        total_records,
        duplicate_key_groups,
        "Number of duplicate business-key groups."
    )

    add_report(
        all_results,
        dataset_name,
        "duplicate_business_key_extra_rows",
        ",".join(key_columns),
        total_records,
        duplicate_extra_rows,
        "Extra rows beyond the first record in each duplicate key group."
    )

    # --------------------------------------------------
    # Check 6: Exact duplicate business records
    # --------------------------------------------------

    business_columns = [
        column
        for column in df.columns
        if column not in METADATA_COLUMNS
    ]

    exact_duplicate_rows = (
        total_records
        - df.select(*business_columns).distinct().count()
    )

    add_report(
        all_results,
        dataset_name,
        "exact_duplicate_business_rows",
        None,
        total_records,
        exact_duplicate_rows,
        "Exact duplicate business records, excluding ingestion metadata."
    )

    print(f"Total records: {total_records}")
    print(f"Null business-key records: {null_key_records}")
    print(f"Duplicate key groups: {duplicate_key_groups}")
    print(f"Extra duplicate key rows: {duplicate_extra_rows}")
    print(f"Exact duplicate rows: {exact_duplicate_rows}")


## 7. Create completeness and uniqueness report

report_df = spark.createDataFrame(
    all_results,
    schema=report_schema
)

display(
    report_df.orderBy(
        "dataset_name",
        "check_type",
        "column_name"
    )
)


## 8. Save report as Delta table

(
    report_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(REPORT_TABLE)
)

print(f"Report saved: {REPORT_TABLE}")


## 9. Calculate dataset completeness percentage

completeness_df = (
    report_df
    .filter(
        F.col("check_type") == "null_and_blank_check"
    )
    .groupBy("dataset_name")
    .agg(
        F.sum("total_records").alias("total_required_values"),

        F.sum("failed_records").alias("missing_required_values")
    )
    .withColumn(
        "completeness_percentage",
        F.when(
            F.col("total_required_values") > 0,
            F.round(
                (
                    F.col("total_required_values")
                    - F.col("missing_required_values")
                )
                / F.col("total_required_values") * 100,
                2
            )
        ).otherwise(0)
    )
    .orderBy("dataset_name")
)

display(completeness_df)


## 10. Dataset uniqueness summary

uniqueness_df = (
    report_df
    .filter(
        F.col("check_type").isin(
            "duplicate_business_key_groups",
            "duplicate_business_key_extra_rows",
            "exact_duplicate_business_rows"
        )
    )
    .groupBy("dataset_name")
    .pivot(
        "check_type",
        [
            "duplicate_business_key_groups",
            "duplicate_business_key_extra_rows",
            "exact_duplicate_business_rows"
        ]
    )
    .agg(
        F.max("failed_records")
    )
    .fillna(0)
    .orderBy("dataset_name")
)

display(uniqueness_df)


## 11. Overall quality summary

total_checks = report_df.count()

passed_checks = (
    report_df
    .filter(F.col("status") == "PASS")
    .count()
)

failed_checks = (
    report_df
    .filter(F.col("status") == "FAIL")
    .count()
)

print("=" * 60)
print("COMPLETENESS AND UNIQUENESS SUMMARY")
print("=" * 60)

print(f"Total checks: {total_checks}")
print(f"Passed checks: {passed_checks}")
print(f"Failed checks: {failed_checks}")

print("=" * 60)


## 12. Final report

display(
    spark.table(REPORT_TABLE)
    .orderBy(
        "dataset_name",
        "check_type",
        "column_name"
    )
)

print("=" * 60)
print("COMPLETENESS AND UNIQUENESS CHECKS COMPLETED")
print("=" * 60)

print(f"Report table: {REPORT_TABLE}")

print("=" * 60)



## Completeness and Uniqueness – Execution Summary


| Metric | Result |
|---|---:|
| Total Checks | 81 |
| Passed Checks | 81 |
| Failed Checks | 0 |
| **Success Rate** | **100%** |


### Execution Status


```text
┌──────────────────────────────────────────────┐
│       COMPLETENESS & UNIQUENESS              │
│                 COMPLETED                    │
├──────────────────────────────────────────────┤
│  Total Checks          : 81                  │
│  Passed Checks         : 81                  │
│  Failed Checks         : 0                   │
│                                              │
│  Success Rate          : 100%                │
└──────────────────────────────────────────────┘
```